# Bayesian Modeling of Cross-Country TB Treatment Success

## A Fully Bayesian MCMC Analysis of WHO Data, 2012–2023

> **Course:** Fundamentals of Statistical Learning II — M.Sc. in Data Science, a.y. 2025–2026

**Architecture:**
- **Code:** Stored in GitHub — edit locally, push to GitHub
- **Data/Runs/Outputs:** Stored in Google Drive — persists across Colab sessions
- **Runtime:** R (via IRkernel in Colab)

---
## Phase 0 / Section A — Project Infrastructure & Reproducibility Setup

> **Goal:** Establish a clean, reproducible project scaffold before any analysis.
>
> Steps: 0.1 Directory structure · 0.2 Script pipeline · 0.3 Software stack & seed

In [ ]:
# A0) Detect environment (Colab vs local) & mount Drive if needed

IS_COLAB <- dir.exists("/content")

if (IS_COLAB) {
  if (!dir.exists("/content/drive")) {
    system("echo 'Please mount Google Drive from the sidebar (folder icon → Mount Drive)'")
  } else {
    cat("Google Drive already mounted at /content/drive\n")
  }
} else {
  cat("Running locally (not on Colab)\n")
}

In [ ]:
# A1) Define all project paths (Step 0.1 — directory structure)
#
# Two roots: CODE_ROOT (repo/notebook code) and STORAGE_ROOT (data/outputs).
# On Colab these differ (repo clone vs Drive). Locally they are the same.
# All downstream paths are derived from these two roots only.

if (IS_COLAB) {
  CODE_ROOT    <- file.path("/content", "FSL_2_Final_Project")
  STORAGE_ROOT <- file.path("/content/drive/MyDrive/Projects", "FSL_2_Final_Project")
} else {
  # Locally: find the project root (parent of notebooks/)
  PROJECT_ROOT <- normalizePath(file.path(getwd(), ".."), mustWork = FALSE)
  if (!file.exists(file.path(PROJECT_ROOT, "notebooks", "main.ipynb"))) {
    if (file.exists(file.path(getwd(), "notebooks", "main.ipynb"))) {
      PROJECT_ROOT <- getwd()
    }
  }
  CODE_ROOT    <- PROJECT_ROOT
  STORAGE_ROOT <- PROJECT_ROOT
}

# --- Canonical directory names (matching TODO plan exactly) ---
DATA_RAW     <- file.path(STORAGE_ROOT, "data_raw")
DATA_PROC    <- file.path(STORAGE_ROOT, "data_processed")
RUNS_DIR     <- file.path(STORAGE_ROOT, "runs")
OUT_DIR      <- file.path(STORAGE_ROOT, "outputs")
OUT_FIG      <- file.path(OUT_DIR, "figures")
OUT_TAB      <- file.path(OUT_DIR, "tables")
OUT_MODELS   <- file.path(OUT_DIR, "model_objects")
OUT_DIAG     <- file.path(OUT_DIR, "diagnostics")
OUT_SIM      <- file.path(OUT_DIR, "simulations")
MODELS_DIR   <- file.path(CODE_ROOT, "models")
REPORT_DIR   <- file.path(CODE_ROOT, "report")
SCRIPTS_DIR  <- file.path(CODE_ROOT, "scripts")
NOTES_DIR    <- file.path(CODE_ROOT, "notes")

# Create all directories (no-op if they already exist)
for (p in c(DATA_RAW, DATA_PROC, RUNS_DIR,
            OUT_FIG, OUT_TAB, OUT_MODELS, OUT_DIAG, OUT_SIM,
            MODELS_DIR, REPORT_DIR, SCRIPTS_DIR, NOTES_DIR)) {
  dir.create(p, recursive = TRUE, showWarnings = FALSE)
}

cat("CODE_ROOT:   ", CODE_ROOT, "\n")
cat("STORAGE_ROOT:", STORAGE_ROOT, "\n\n")
cat("DATA_RAW:    ", DATA_RAW, "\n")
cat("DATA_PROC:   ", DATA_PROC, "\n")
cat("OUT_FIG:     ", OUT_FIG, "\n")
cat("OUT_TAB:     ", OUT_TAB, "\n")
cat("OUT_MODELS:  ", OUT_MODELS, "\n")
cat("OUT_DIAG:    ", OUT_DIAG, "\n")
cat("OUT_SIM:     ", OUT_SIM, "\n")
cat("MODELS_DIR:  ", MODELS_DIR, "\n")
cat("REPORT_DIR:  ", REPORT_DIR, "\n")
cat("NOTES_DIR:   ", NOTES_DIR, "\n")

In [ ]:
# A2) Clone/Update code from GitHub (Colab only) + log Git SHA (note 6)

REPO_OWNER <- "armanfeili"
REPO_NAME  <- "FSL_2_Final_Project"
REPO_URL   <- paste0("https://github.com/", REPO_OWNER, "/", REPO_NAME, ".git")

if (IS_COLAB) {
  if (dir.exists(CODE_ROOT)) {
    cat(CODE_ROOT, "exists -> pulling latest...\n")
    system2("git", c("-C", CODE_ROOT, "fetch", "--prune"))
    system2("git", c("-C", CODE_ROOT, "checkout", "main"))
    system2("git", c("-C", CODE_ROOT, "pull", "--ff-only"))
  } else {
    cat("Cloning", REPO_URL, "->", CODE_ROOT, "...\n")
    system2("git", c("clone", "--depth=1", REPO_URL, CODE_ROOT))
  }
  cat("Repository ready at:", CODE_ROOT, "\n")
} else {
  cat("Running locally -- no clone needed.\n")
}

# Log Git metadata for reproducibility
git_root <- if (IS_COLAB) CODE_ROOT else STORAGE_ROOT
git_sha    <- tryCatch(system2("git", c("-C", git_root, "rev-parse", "HEAD"),
                               stdout = TRUE, stderr = FALSE), error = function(e) "unknown")
git_branch <- tryCatch(system2("git", c("-C", git_root, "rev-parse", "--abbrev-ref", "HEAD"),
                               stdout = TRUE, stderr = FALSE), error = function(e) "unknown")

git_info <- list(
  repo_url   = REPO_URL,
  branch     = git_branch,
  commit_sha = git_sha,
  timestamp  = format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z")
)

cat("\nGit metadata:\n")
cat("  Repo:   ", git_info$repo_url, "\n")
cat("  Branch: ", git_info$branch, "\n")
cat("  SHA:    ", git_info$commit_sha, "\n")
cat("  Time:   ", git_info$timestamp, "\n")

In [ ]:
# A3) Install system JAGS first, then R packages (Step 0.3)
#
# IMPORTANT: JAGS system library must be present before rjags is installed,
# otherwise rjags compilation will fail on Colab/Linux.

# --- 1. Install JAGS system library (Colab / Linux only) ---
if (IS_COLAB) {
  if (system("which jags", ignore.stdout = TRUE, ignore.stderr = TRUE) != 0) {
    cat("Installing JAGS system library...\n")
    system("apt-get update -qq && apt-get install -y -qq jags", ignore.stdout = TRUE)
    cat("JAGS installed.\n")
  } else {
    cat("JAGS already installed.\n")
  }
}

# --- 2. Install R packages ---
required_packages <- c(
  # MCMC & Bayesian
  "rjags",        # Interface to JAGS (needs system JAGS)
  "coda",         # MCMC diagnostics
  "MCMCvis",      # MCMC visualization
  "bayesplot",    # Bayesian diagnostic plots
  # Data wrangling
  "tidyverse",    # dplyr, ggplot2, tidyr, readr, stringr, forcats, etc.
  "data.table",   # Fast data manipulation
  # Visualization
  "patchwork",    # Combine plots
  "corrplot",     # Correlation matrices
  # Frequentist comparison
  "lme4",         # Mixed-effects models
  "VGAM",         # Beta-binomial regression
  "aod",          # Analysis of overdispersed data
  # Utilities
  "yaml",         # Config files
  "knitr",        # Tables
  "car"           # VIF
)

installed <- rownames(installed.packages())
to_install <- setdiff(required_packages, installed)

if (length(to_install) > 0) {
  cat("Installing:", paste(to_install, collapse = ", "), "\n")
  install.packages(to_install, repos = "https://cloud.r-project.org", quiet = TRUE)
} else {
  cat("All R packages already installed.\n")
}

cat("Package installation complete.\n")

In [ ]:
# A4) Load all libraries

suppressPackageStartupMessages({
  library(rjags)
  library(coda)
  library(MCMCvis)
  library(bayesplot)
  library(tidyverse)
  library(data.table)
  library(patchwork)
  library(corrplot)
  library(lme4)
  library(VGAM)
  library(aod)
  library(yaml)
  library(knitr)
  library(car)
})

cat("Libraries loaded.\n")
cat("R version:", R.version.string, "\n")

jags_ver <- tryCatch(
  system("jags --version 2>&1 | head -1", intern = TRUE),
  error = function(e) "JAGS version unknown"
)
cat("JAGS:     ", jags_ver, "\n")

In [ ]:
# A5) Set seed, ggplot2 theme, and record version manifest (Step 0.3)

SEED <- 2026
set.seed(SEED)

# Common ggplot2 theme
theme_set(theme_minimal(base_size = 13))

cat("Seed set to", SEED, "\n")
cat("ggplot2 theme set to theme_minimal(base_size = 13)\n")

# --- Version manifest (for reproducibility appendix) ---
pkg_list <- c("rjags", "coda", "MCMCvis", "bayesplot",
              "dplyr", "ggplot2", "tidyr", "readr", "stringr", "forcats",
              "data.table", "patchwork", "corrplot",
              "lme4", "VGAM", "aod",
              "yaml", "knitr", "car")

version_manifest <- data.frame(
  package = pkg_list,
  version = sapply(pkg_list, function(p) as.character(packageVersion(p))),
  stringsAsFactors = FALSE,
  row.names = NULL
)

# Add R and JAGS rows at the top
version_manifest <- rbind(
  data.frame(package = "R", version = paste(R.version$major, R.version$minor, sep = ".")),
  data.frame(package = "JAGS", version = tryCatch(
    gsub("[^0-9.]", "", system("jags --version 2>&1 | head -1", intern = TRUE)),
    error = function(e) "unknown"
  )),
  version_manifest
)

# Save version manifest
manifest_path <- file.path(OUT_TAB, "version_manifest.csv")
write.csv(version_manifest, manifest_path, row.names = FALSE)

# Save git metadata alongside version manifest
git_info_path <- file.path(OUT_TAB, "git_metadata.yaml")
write_yaml(git_info, git_info_path)

# Save setup metadata (seed, roots, canonical paths)
setup_meta <- list(
  seed         = SEED,
  CODE_ROOT    = CODE_ROOT,
  STORAGE_ROOT = STORAGE_ROOT,
  paths = list(
    DATA_RAW   = DATA_RAW,
    DATA_PROC  = DATA_PROC,
    OUT_FIG    = OUT_FIG,
    OUT_TAB    = OUT_TAB,
    OUT_MODELS = OUT_MODELS,
    OUT_DIAG   = OUT_DIAG,
    OUT_SIM    = OUT_SIM,
    MODELS_DIR = MODELS_DIR,
    REPORT_DIR = REPORT_DIR,
    SCRIPTS_DIR = SCRIPTS_DIR,
    NOTES_DIR  = NOTES_DIR,
    RUNS_DIR   = RUNS_DIR
  ),
  timestamp = format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z")
)

setup_meta_path <- file.path(OUT_TAB, "setup_metadata.yaml")
write_yaml(setup_meta, setup_meta_path)

cat("\nVersion manifest saved to: ", manifest_path, "\n")
cat("Git metadata saved to:    ", git_info_path, "\n")
cat("Setup metadata saved to:  ", setup_meta_path, "\n")
print(version_manifest)

In [ ]:
# A6) Verify raw data files & required Phase 0 files (Steps 0.1)

# --- Strict raw-file verification ---
expected_raw_files <- c(
  "TB_outcomes_2026-04-04.csv",
  "TB_burden_countries_2026-04-04.csv",
  "TB_data_dictionary_2026-04-04.csv"
)

actual_raw_files <- list.files(DATA_RAW, pattern = "\\.csv$")

cat("=== Raw data verification ===\n")
cat("DATA_RAW:", DATA_RAW, "\n\n")

# Check each expected file
all_found <- TRUE
for (f in expected_raw_files) {
  path <- file.path(DATA_RAW, f)
  if (file.exists(path)) {
    info <- file.info(path)
    cat(sprintf("  [OK]      %s  (%.1f KB)\n", f, info$size / 1024))
  } else {
    cat(sprintf("  [MISSING] %s\n", f))
    all_found <- FALSE
  }
}

# Warn about extra CSVs
extra_files <- setdiff(actual_raw_files, expected_raw_files)
if (length(extra_files) > 0) {
  cat("\n  [INFO] Extra CSV files in DATA_RAW (not expected by main analysis):\n")
  for (f in extra_files) {
    cat(sprintf("    - %s\n", f))
  }
}

if (!all_found) {
  stop("One or more required raw data files are missing from DATA_RAW. Cannot proceed.")
}
cat("\nAll 3 required raw files present.\n")

# --- Ensure required Phase 0 files/dirs exist ---
# notes/decision_log.md
decision_log_path <- file.path(NOTES_DIR, "decision_log.md")
if (!file.exists(decision_log_path)) {
  writeLines(c(
    "# Decision Log",
    "",
    "> All frozen choices for the Bayesian TB Treatment Success project are recorded here.",
    "> Each entry includes the date, what was decided, and why.",
    "",
    "---",
    "",
    "## Decisions",
    "",
    "*(No decisions recorded yet.)*"
  ), decision_log_path)
  cat("Created:", decision_log_path, "\n")
} else {
  cat("Exists: ", decision_log_path, "\n")
}

# Confirm report/ and scripts/ directories exist
cat("Exists: ", REPORT_DIR, " ->", dir.exists(REPORT_DIR), "\n")
cat("Exists: ", SCRIPTS_DIR, " ->", dir.exists(SCRIPTS_DIR), "\n")
cat("Exists: ", MODELS_DIR, " ->", dir.exists(MODELS_DIR), "\n")

### Step 0.2 — Execution Pipeline

> **This notebook (`notebooks/main.ipynb`) is the sole execution source of truth.**
> No separate numbered `.R` scripts are used to run the analysis. The TODO plan's
> script pipeline table describes the *logical* stages; each stage maps to a
> notebook section below. The `scripts/` directory is reserved for optional
> helper utilities only — it is never the primary execution path.

| # | Logical Stage | Notebook Section | Purpose | Inputs | Outputs |
|---|--------------|------------------|---------|--------|---------|
| 00 | Setup | **A** (this section) | Load packages, set seed, define paths, helpers | — | Environment ready |
| 01 | Load & inspect data | **B0** | Import raw CSVs, audit dimensions & keys | `data_raw/` CSVs | `intake_summary.csv` |
| 02 | Build main analysis table | **B1–B2** | Merge, filter, standardize, lock dataset | `data_raw/` CSVs | `main_analysis_table_locked.csv/.rds` |
| 03 | EDA | **C** | All exploratory plots & tables | Locked table | Figures + tables |
| 04 | Prior predictive checks | **D** (pre-fit) | Simulate from priors, verify plausibility | Locked table | Prior predictive plots |
| 05 | Fit M1 (binomial) | **D1** | JAGS fit for Model 1 | Locked table | Posterior draws `.rds` |
| 06 | Fit M2 (beta-binomial) | **D2** | JAGS fit for Model 2 | Locked table | Posterior draws `.rds` |
| 07 | Fit M3 (hierarchical) | **D3** | JAGS fit for Model 3 | Locked table | Posterior draws `.rds` |
| 08 | MCMC diagnostics | **E** | Trace plots, R-hat, ESS, convergence tests | Posterior draws | Diagnostic figures + tables |
| 09 | Posterior inference | **F0** | Summaries, intervals, directional probabilities | Posterior draws | Inference tables |
| 10 | Posterior predictive checks | **F1** | Y_rep, test quantities, Bayesian p-values | Posterior draws + locked table | PPC figures + tables |
| 11 | Parameter recovery | **G** | Simulate, refit, coverage/bias | Locked table + true params | Recovery tables + plots |
| 12 | DIC comparison | **F2** | Observed-data log-likelihood DIC | Posterior draws + locked table | DIC table |
| 13 | Frequentist comparison | **H** | GLM, VGAM, GLMM analogues | Locked table | Comparison table |
| 14 | Sensitivity analyses | **I** | 5 robustness checks | Various | Sensitivity tables |
| 15 | Polish outputs | **J** | Final report-ready assets | All outputs | Polished figures/tables |
| 16 | Report support outputs | **J** | Export final numbers for abstract/appendix | All outputs | Summary file |

**Rule:** Downstream sections read frozen exported objects (locked table, posterior `.rds` files) — they never silently rebuild upstream outputs.

In [ ]:
# A7) Helper functions (Step 0.3 — part of setup)

# Ensure a directory exists (creates recursively if needed)
ensure_dir <- function(path) {
  dir.create(path, recursive = TRUE, showWarnings = FALSE)
  invisible(path)
}

# Save a ggplot figure to the outputs/figures directory
save_fig <- function(plot, name, width = 8, height = 5, dpi = 300) {
  path <- file.path(OUT_FIG, paste0(name, ".png"))
  ggsave(path, plot, width = width, height = height, dpi = dpi)
  cat("Saved figure:", path, "\n")
  invisible(path)
}

# Save a data frame as CSV to the outputs/tables directory
save_table <- function(df, name) {
  path <- file.path(OUT_TAB, paste0(name, ".csv"))
  write.csv(df, path, row.names = FALSE)
  cat("Saved table:", path, "\n")
  invisible(path)
}

# Log a runtime event
log_runtime <- function(event, start_time = NULL) {
  elapsed <- if (!is.null(start_time)) {
    sprintf("%.1f sec", as.numeric(difftime(Sys.time(), start_time, units = "secs")))
  } else {
    ""
  }
  cat(sprintf("[%s] %s %s\n", format(Sys.time(), "%H:%M:%S"), event, elapsed))
}

cat("Helper functions defined: ensure_dir(), save_fig(), save_table(), log_runtime()\n")

---
## Phase 2–3 / Section B — Data Loading & Cleaning

In [ ]:
# B0) Load raw CSV data from the single canonical DATA_RAW directory

cat("Loading data from:", DATA_RAW, "\n\n")

outcomes  <- read_csv(file.path(DATA_RAW, "TB_outcomes_2026-04-04.csv"),
                      show_col_types = FALSE)
burden    <- read_csv(file.path(DATA_RAW, "TB_burden_countries_2026-04-04.csv"),
                      show_col_types = FALSE)
data_dict <- read_csv(file.path(DATA_RAW, "TB_data_dictionary_2026-04-04.csv"),
                      show_col_types = FALSE)

cat("Data loaded:\n")
cat("  outcomes:", nrow(outcomes), "rows x", ncol(outcomes), "cols\n")
cat("  burden:  ", nrow(burden),  "rows x", ncol(burden),  "cols\n")
cat("  data_dict:", nrow(data_dict), "rows x", ncol(data_dict), "cols\n")

In [ ]:
# B1) Data Cleaning Pipeline (per PROJECT_PLAN.md §6)
#
# Steps:
# 1. Keep needed columns
# 2. Construct success = newrel_succ, cohort = newrel_coh
# 3. Restrict to 2012–2023
# 4. Apply inclusion flag: rel_with_new_flg == 1
# 5. Drop invalid rows
# 6. Merge outcomes + burden by (iso3, year)
# 7. Standardize continuous predictors
# 8. Encode g_whoregion as factor
# 9. Lock main-analysis table

# TODO: Implement cleaning pipeline
cat("⏳ Data cleaning pipeline — implement per PROJECT_PLAN.md §6\n")

In [ ]:
# B2) Sample Attrition Table

# TODO: Fill with actual counts after cleaning
# attrition <- tribble(
#   ~Stage,                                ~Rows, ~Countries, ~Years,
#   "Raw merged rows",                       NA,       NA,      NA,
#   "After year restriction (2012–2023)",     NA,       NA,      NA,
#   "After comparability flag",               NA,       NA,      NA,
#   "After missingness removal",              NA,       NA,      NA,
#   "After cohort filter (≥50)",              NA,       NA,      NA
# )
# kable(attrition)

cat("⏳ Attrition table — fill after implementing cleaning\n")

---
## Section C — Exploratory Data Analysis

In [ ]:
# C0) EDA — Descriptive summaries (per PROJECT_PLAN.md §7)
#
# TODO:
# 1. Sample size summary
# 2. Cohort distribution histogram
# 3. Success rate distribution
# 4. Temporal trends by WHO region
# 5. Bivariate relationships
# 6. Correlation matrix & VIF screening
# 7. Country-level spread

cat("⏳ EDA section — implement per PROJECT_PLAN.md §7\n")

---
## Section D — MCMC Modeling with JAGS

In [ ]:
# D0) MCMC Run Configuration

RUN_ID <- paste0(format(Sys.time(), "%Y-%m-%d_%H-%M"), "_bayesian_tb")
RUN_DIR <- file.path(RUNS_DIR, RUN_ID)

for (subdir in c("mcmc_output", "plots", "diagnostics", "tables")) {
  dir.create(file.path(RUN_DIR, subdir), recursive = TRUE, showWarnings = FALSE)
}

# MCMC settings
mcmc_cfg <- list(
  n_chains   = 4,
  n_burnin   = 4000,
  n_iter     = 8000,
  n_thin     = 1,
  seed       = SEED
)

# Save config
write_yaml(mcmc_cfg, file.path(RUN_DIR, "mcmc_config.yaml"))

cat("🏷️ RUN_ID: ", RUN_ID, "\n")
cat("📁 RUN_DIR:", RUN_DIR, "\n")
cat("✅ MCMC config saved\n")

In [ ]:
# D1) Model 1 — Binomial Logistic Regression (Baseline)

model1_string <- "
model {
  for (i in 1:N) {
    logit(p[i]) <- beta0 + inprod(X[i,], beta[]) + gamma[region[i]]
    Y[i] ~ dbin(p[i], n[i])
  }

  # Priors
  beta0 ~ dnorm(0, 1/(2.5*2.5))
  for (j in 1:p) {
    beta[j] ~ dnorm(0, 1/(2.5*2.5))
  }
  for (r in 2:R) {
    gamma[r] ~ dnorm(0, 1/(2.5*2.5))
  }
  gamma[1] <- 0  # baseline region
}
"

cat("✅ Model 1 (Binomial Logistic) defined\n")

# TODO: Prepare data list, compile & run JAGS model
# jags_data <- list(Y = ..., n = ..., X = ..., region = ..., N = ..., p = ..., R = ...)
# model1 <- jags.model(textConnection(model1_string), data = jags_data,
#                       n.chains = mcmc_cfg$n_chains, n.adapt = 1000)
# update(model1, mcmc_cfg$n_burnin)
# samples1 <- coda.samples(model1, variable.names = c("beta0", "beta", "gamma"),
#                           n.iter = mcmc_cfg$n_iter, thin = mcmc_cfg$n_thin)

In [ ]:
# D2) Model 2 — Beta-Binomial Regression

model2_string <- "
model {
  for (i in 1:N) {
    # Logit link for mean
    logit(mu[i]) <- beta0 + inprod(X[i,], beta[]) + gamma[region[i]]

    # Beta-distributed latent probability
    alpha[i] <- mu[i] * phi
    betap[i] <- (1 - mu[i]) * phi
    theta[i] ~ dbeta(alpha[i], betap[i])

    # Observed successes
    Y[i] ~ dbin(theta[i], n[i])
  }

  # Priors
  beta0 ~ dnorm(0, 1/(2.5*2.5))
  for (j in 1:p) {
    beta[j] ~ dnorm(0, 1/(2.5*2.5))
  }
  for (r in 2:R) {
    gamma[r] ~ dnorm(0, 1/(2.5*2.5))
  }
  gamma[1] <- 0

  phi ~ dgamma(2, 0.1)
}
"

cat("✅ Model 2 (Beta-Binomial) defined\n")

# TODO: Compile & run JAGS model
# model2 <- jags.model(textConnection(model2_string), data = jags_data,
#                       n.chains = mcmc_cfg$n_chains, n.adapt = 1000)
# update(model2, mcmc_cfg$n_burnin)
# samples2 <- coda.samples(model2,
#   variable.names = c("beta0", "beta", "gamma", "phi"),
#   n.iter = mcmc_cfg$n_iter, thin = mcmc_cfg$n_thin)

In [ ]:
# D3) Model 3 — Hierarchical Beta-Binomial Regression

model3_string <- "
model {
  for (i in 1:N) {
    logit(mu[i]) <- beta0 + inprod(X[i,], beta[]) + gamma[region[i]] + u[country[i]]

    alpha[i] <- mu[i] * phi
    betap[i] <- (1 - mu[i]) * phi
    theta[i] ~ dbeta(alpha[i], betap[i])

    Y[i] ~ dbin(theta[i], n[i])
  }

  # Country random effects
  for (c in 1:C) {
    u[c] ~ dnorm(0, tau_u)
  }
  tau_u <- 1 / (sigma_u * sigma_u)
  sigma_u ~ dnorm(0, 1) T(0, )  # Half-Normal(0,1)

  # Fixed-effect priors
  beta0 ~ dnorm(0, 1/(2.5*2.5))
  for (j in 1:p) {
    beta[j] ~ dnorm(0, 1/(2.5*2.5))
  }
  for (r in 2:R) {
    gamma[r] ~ dnorm(0, 1/(2.5*2.5))
  }
  gamma[1] <- 0

  phi ~ dgamma(2, 0.1)
}
"

cat("✅ Model 3 (Hierarchical Beta-Binomial) defined\n")

# TODO: Compile & run JAGS model
# jags_data3 <- c(jags_data, list(country = ..., C = ...))
# model3 <- jags.model(textConnection(model3_string), data = jags_data3,
#                       n.chains = mcmc_cfg$n_chains, n.adapt = 1000)
# update(model3, mcmc_cfg$n_burnin)
# samples3 <- coda.samples(model3,
#   variable.names = c("beta0", "beta", "gamma", "phi", "sigma_u"),
#   n.iter = mcmc_cfg$n_iter, thin = mcmc_cfg$n_thin)

---
## Section E — MCMC Diagnostics

In [ ]:
# E0) MCMC Diagnostics (per PROJECT_PLAN.md §11)
#
# For each model:
# 1. Trace plots
# 2. Posterior density plots
# 3. Autocorrelation plots
# 4. R-hat (Gelman-Rubin)
# 5. Effective sample size (ESS)
# 6. Geweke diagnostics

# Example (uncomment after fitting models):
# summary(samples1)
# gelman.diag(samples1)
# effectiveSize(samples1)
# traceplot(samples1)

cat("⏳ MCMC diagnostics — run after fitting models\n")

---
## Section F — Posterior Inference & Model Comparison

In [ ]:
# F0) Posterior Inference (per PROJECT_PLAN.md §13)
#
# For each model:
# 1. Posterior means & medians
# 2. 95% equal-tail credible intervals
# 3. 95% HPD intervals
# 4. Posterior probabilities of directional effects

# Example:
# MCMCsummary(samples1, round = 4)
# HPDinterval(samples1)

cat("⏳ Posterior inference — implement after fitting models\n")

In [ ]:
# F1) Posterior Predictive Checks (per PROJECT_PLAN.md §14)
#
# Test quantities:
# T1: Mean treatment success rate
# T2: Variance of success rates (key diagnostic)
# T3: Count of low-success country-years
# T4: Within-region variance

cat("⏳ Posterior predictive checks — implement after fitting models\n")

In [ ]:
# F2) Model Comparison via DIC (per PROJECT_PLAN.md §15)
#
# IMPORTANT: Compute observed-data DIC (beta-binomial log-PMF)
# for Models 2 & 3, not JAGS's default conditional DIC.

# Example:
# dic1 <- dic.samples(model1, n.iter = 5000, type = "pD")
# For Models 2 & 3: compute manually from posterior draws

cat("⏳ DIC model comparison — implement after fitting models\n")

---
## Section G — Parameter Recovery Simulation

In [ ]:
# G0) Parameter Recovery (per PROJECT_PLAN.md §12)
#
# For each model:
# 1. Choose true parameter values (from posteriors)
# 2. Simulate 50 datasets (same design structure)
# 3. Refit model to each synthetic dataset
# 4. Evaluate: bias, RMSE, coverage

cat("⏳ Parameter recovery simulation — implement after main analysis\n")

---
## Section H — Frequentist Comparison (Bonus)

In [ ]:
# H0) Frequentist Comparison (per PROJECT_PLAN.md §16)
#
# M1 freq: glm(cbind(success, cohort - success) ~ ..., family = binomial)
# M2 freq: VGAM::vglm(..., family = betabinomial)
# M3 freq: lme4::glmer(cbind(success, cohort - success) ~ ... + (1|country), family = binomial)

cat("⏳ Frequentist comparison — implement after main Bayesian analysis\n")

---
## Section I — Sensitivity Analyses

In [ ]:
# I0) Sensitivity Analyses (per PROJECT_PLAN.md §17)
#
# 1. cohort ≥ 50 vs. cohort > 0
# 2. Main predictors vs. + e_tbhiv_prct
# 3. phi prior: Gamma(2,0.1) vs. Gamma(1,0.1)
# 4. sigma_u prior: HalfNormal(0,1) vs. HalfNormal(0,2.5)
# 5. Post-2021 definitional check (used_2021_defs_flg == 1)

cat("⏳ Sensitivity analyses — implement after main results confirmed\n")

---
## Section J — Save Results

In [ ]:
# J0) Save all outputs to Drive

cat("\n" , paste(rep("=", 60), collapse = ""), "\n")
cat("  RUN COMPLETE\n")
cat(paste(rep("=", 60), collapse = ""), "\n\n")
cat("📁 All outputs saved to Drive:\n")
cat("   ", RUN_DIR, "\n")
cat("\n📋 Session Info:\n")
sessionInfo()